In [4]:
# Preparing the code for a full SimCLR pipeline using PyTorch and timm with ConvNeXt V2 as the backbone.

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
import random

# Define the custom Dataset for bi-planar X-rays
class BiPlanarXrayDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.root_dir = root_dir
        self.transform = transform
        for pid in sorted(os.listdir(root_dir)):
            for side in ['left', 'right']:
                la_path = os.path.join(root_dir, pid, 'DRR', side, 'la.png')
                pa_path = os.path.join(root_dir, pid, 'DRR', side, 'pa.png')
                if os.path.exists(la_path) and os.path.exists(pa_path):
                    self.samples.append((la_path, pa_path))


    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        la_path, pa_path = self.samples[idx]
        la_img = Image.open(la_path).convert('L')
        pa_img = Image.open(pa_path).convert('L')
        if self.transform:
            la_view1 = self.transform(la_img)
            la_view2 = self.transform(la_img)
            pa_view1 = self.transform(pa_img)
            pa_view2 = self.transform(pa_img)
        else:
            la_view1 = la_view2 = la_img
            pa_view1 = pa_view2 = pa_img

        # Return 2 positive pairs: (la1, la2) and (pa1, pa2)
        return la_view1, la_view2, pa_view1, pa_view2

# Data augmentations for SimCLR
simclr_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# SimCLR model definition
class SimCLRModel(nn.Module):
    def __init__(self, backbone_name="convnextv2_tiny", projection_dim=128):
        super(SimCLRModel, self).__init__()
        self.encoder = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        feature_dim = self.encoder.num_features
        self.projector = nn.Sequential(
            nn.Linear(feature_dim, feature_dim),
            nn.ReLU(),
            nn.Linear(feature_dim, projection_dim)
        )

    def forward(self, x):
        features = self.encoder(x)
        projections = self.projector(features)
        projections = F.normalize(projections, dim=1)
        return projections

# Contrastive Loss (NT-Xent)
def nt_xent_loss(z1, z2, temperature=0.5):
    batch_size = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    sim_matrix = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=2)
    sim_matrix = sim_matrix / temperature

    labels = torch.arange(batch_size).to(z.device)
    labels = torch.cat([labels, labels], dim=0)

    mask = torch.eye(2 * batch_size, dtype=torch.bool).to(z.device)
    sim_matrix = sim_matrix.masked_fill(mask, -9e15)

    loss = F.cross_entropy(sim_matrix, labels)
    return loss

# DataLoader setup
def get_dataloader(data_path, batch_size=16):
    dataset = BiPlanarXrayDataset(data_path, transform=simclr_transform)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=4)

# Training loop
def train_simclr(model, dataloader, optimizer, device, epochs=10):
    model.to(device)
    model.train()

    for epoch in range(epochs):
        total_loss = 0.0
        for la1, la2, pa1, pa2 in dataloader:
            la1, la2 = la1.to(device), la2.to(device)
            pa1, pa2 = pa1.to(device), pa2.to(device)

            # Use both LA and PA pairs
            z1 = model(la1)
            z2 = model(la2)
            z3 = model(pa1)
            z4 = model(pa2)

            loss = nt_xent_loss(z1, z2) + nt_xent_loss(z3, z4)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}")

    torch.save(model.encoder.state_dict(), "convnextv2_finetuned_simclr.pth")
    print("Model saved!")

# Wrap everything in a function to avoid running at import
def main():
    data_path = "../data/processed"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SimCLRModel()
    dataloader = get_dataloader(data_path)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
    train_simclr(model, dataloader, optimizer, device)

main()



Traceback (most recent call last):
  File "<string>", line 1, in <module>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/3d-recon-ai/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
  File "<string>", line 1, in <module>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/3d-recon-ai/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
  File "<string>", line 1, in <module>
  File "/opt/anaconda3/envs/3d-recon-ai/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/opt/anaconda3/envs/3d-recon-ai/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
    exitcode = _main(fd, parent_sentinel)
    exitcode = _main(fd, parent_sentinel)
        exitcode = _main(fd, parent_sentinel)
                         ^ ^         ^     ^ ^ ^ ^ ^^^^   ^ ^ ^ ^ ^^^^^^^^ ^ ^ ^^ ^^^ ^^^^^^^^^^^^^^^^^^^^^^

RuntimeError: DataLoader worker (pid(s) 15726) exited unexpectedly